# EDA de transformación: relevo generacional en manufactura

**Alcance del proyecto.** GEIH de octubre, noviembre y diciembre de 2025. La unidad de análisis es la persona en un mes. El objetivo final es comparar manufactura y demás sectores **del municipio de Cali**, separando empleo asalariado y trabajo por cuenta propia, con tres indicadores ponderados: proporción de personas de 55 años o más, razón de personas de 18 a 34 años por cada persona de 55 años o más y mediana de edad.

**Estado de la fuente:** los seis CSV públicos permiten revisar la transformación y construir un prototipo para **Cali A.M. (Cali y Yumbo)**. No contienen una clave municipal que permita aislar Cali. Por ello, ninguna cifra de este notebook debe presentarse como resultado exclusivo del municipio de Cali. El resultado final queda condicionado al acceso municipal previsto en el documento de alcance.

El notebook lee los productos existentes y no modifica el ETL ni los archivos de datos. Las cifras narradas abajo corresponden a los archivos inspeccionados el 1 de octubre de 2026; ejecute las celdas para actualizar los controles si cambian los insumos.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "data" / "silver").exists():
    raise FileNotFoundError("Abra el notebook desde la raíz del proyecto o desde notebooks/.")

SILVER = ROOT / "data" / "silver"
BRONZE = ROOT / "data" / "bronze"
CLAVES = ["Anio_Encuesta", "Mes_Encuesta", "Id_Vivienda", "Id_Hogar", "Orden_Persona"]

cg = pd.read_csv(SILVER / "caracteristicasgenerales_transformado.csv", low_memory=False).drop(columns="Unnamed: 0", errors="ignore")
ocupados = pd.read_csv(SILVER / "ocupados_trimestre_transformado.csv", low_memory=False).drop(columns="Unnamed: 0", errors="ignore")
df_traducido_clean = pd.read_excel(SILVER / "df_traducido_clean.xlsx")
print("Filas: características generales =", len(cg),
      "| ocupados =", len(ocupados),
      "| final =", len(df_traducido_clean))


Filas: características generales = 203851 | ocupados = 91357 | final = 91357


## 1. Comprobar la unión persona-mes

La clave usada por el ETL es **PER + MES + DIRECTORIO + SECUENCIA_P + ORDEN**, renombrada como las cinco columnas de CLAVES. PER y MES impiden unir observaciones de meses distintos; DIRECTORIO identifica vivienda, SECUENCIA_P identifica hogar y ORDEN a la persona dentro del hogar. **HOGAR/Numero_Hogar se conserva como atributo de control, no como parte de esta clave implementada.** La clave representa una observación persona-mes, no un identificador longitudinal de la misma persona entre meses.

La unión del ETL toma Ocupados como población de referencia. Una comprobación completa requiere: claves no vacías y únicas en ambas fuentes, todos los ocupados con pareja en Características generales, igualdad de atributos compartidos y exactamente las mismas claves en la tabla final.


In [2]:
resumen_union = pd.DataFrame([
    {
        "tabla": nombre,
        "filas": len(tabla),
        "claves_nulas": int(tabla[CLAVES].isna().any(axis=1).sum()),
        "claves_duplicadas": int(tabla.duplicated(CLAVES).sum()),
    }
    for nombre, tabla in [
        ("Características generales", cg),
        ("Ocupados", ocupados),
        ("df_traducido_clean", df_traducido_clean),
    ]
])
display(resumen_union)

cruce_cg = ocupados[CLAVES].merge(
    cg[CLAVES], on=CLAVES, how="left", validate="one_to_one", indicator=True
)
cruce_final = ocupados[CLAVES].merge(
    df_traducido_clean[CLAVES], on=CLAVES, how="outer",
    validate="one_to_one", indicator=True
)
display(pd.DataFrame({
    "ocupados_con_pareja_en_cg": [int(cruce_cg["_merge"].eq("both").sum())],
    "ocupados_sin_pareja_en_cg": [int(cruce_cg["_merge"].eq("left_only").sum())],
    "ocupados_iguales_al_final": [int(cruce_final["_merge"].eq("both").sum())],
    "solo_en_ocupados": [int(cruce_final["_merge"].eq("left_only").sum())],
    "solo_en_final": [int(cruce_final["_merge"].eq("right_only").sum())],
}))

compartidas = [c for c in cg.columns.intersection(ocupados.columns) if c not in CLAVES]
pares = ocupados[CLAVES + compartidas].merge(
    cg[CLAVES + compartidas], on=CLAVES, how="inner",
    suffixes=("_ocupados", "_cg"), validate="one_to_one"
)
discrepancias = {
    c: int((
        pares[c + "_ocupados"].astype("string").fillna("<VACIO>")
        != pares[c + "_cg"].astype("string").fillna("<VACIO>")
    ).sum())
    for c in compartidas
}
display(pd.Series(discrepancias, name="filas_discrepantes").to_frame())
union_correcta = (
    resumen_union[["claves_nulas", "claves_duplicadas"]].to_numpy().sum() == 0
    and cruce_cg["_merge"].eq("both").all()
    and cruce_final["_merge"].eq("both").all()
    and not any(discrepancias.values())
)
print("Unión persona-mes correcta:", union_correcta)


                       tabla   filas  claves_nulas  claves_duplicadas
0  Características generales  203851             0                  0
1                   Ocupados   91357             0                  0
2         df_traducido_clean   91357             0                  0
   ocupados_con_pareja_en_cg  ocupados_sin_pareja_en_cg  ocupados_iguales_al_final  solo_en_ocupados  solo_en_final
0                      91357                          0                      91357                 0              0
                     filas_discrepantes
Periodo_Recoleccion                   0
Numero_Hogar                          0
Area_Metropolitana                    0
Factor_Expansion                      0
Departamento                          0
Unión persona-mes correcta: True


**Resultado observado:** Características generales tiene 203.851 filas; Ocupados y el final, 91.357 cada uno. Hay cero claves nulas y cero duplicados en las tres tablas. Los 91.357 ocupados tienen pareja en Características generales y aparecen exactamente una vez en el final. PERIODO, HOGAR, AREA, FEX_C18 y DPTO coinciden en todos los pares después del renombrado. La tabla final es el subconjunto de ocupados enriquecido con características generales; no representa las 203.851 personas de la muestra completa.


## 2. Acceso municipal y alcance geográfico

La columna AREA del archivo público identifica ciudades o áreas metropolitanas. El código 76 corresponde a **Cali A.M.**, que incluye Cali y Yumbo. DPTO identifica departamento. Ninguna de estas columnas separa los dos municipios. La traducción a la etiqueta "CALI  A.M." tampoco añade detalle municipal. Los valores vacíos de AREA se convierten en "otros" durante la limpieza y tampoco se pueden asignar a Cali.

El acceso licenciado o SPEE con identificador municipal es una **condición de entrada** para el entregable final. Cuando está disponible, se debe validar que el código del municipio está presente, filtrar Cali y repetir la comprobación de unión y los indicadores. No se debe inferir municipio a partir de nombres de área metropolitana, vivienda o departamento.


In [3]:
archivos_fuente = [BRONZE / (modulo + "_" + mes + ".CSV") for modulo in ("caracteristicasgenerales", "ocupados") for mes in ("oct", "nov", "dic")]
columnas_fuente = {
    p.name: pd.read_csv(p, nrows=0, encoding="latin-1").columns.tolist()
    for p in archivos_fuente
}
candidatas_municipales = {
    nombre: [c for c in columnas if any(t in c.upper() for t in ("MPIO", "MUNICIP", "COD_MUN"))]
    for nombre, columnas in columnas_fuente.items()
}
display(pd.DataFrame([
    {"archivo": nombre, "columnas_municipales_candidatas": ", ".join(candidatas) or "ninguna"}
    for nombre, candidatas in candidatas_municipales.items()
]))
print("Columna municipal en el final:", [
    c for c in df_traducido_clean.columns
    if any(t in c.upper() for t in ("MPIO", "MUNICIP", "COD_MUN"))
])
print("Registros de Cali A.M. en el final:",
      int(df_traducido_clean["Area_Metropolitana"].eq("CALI  A.M.").sum()))
print("AREA sin identificación específica, etiquetada como otros:",
      int(df_traducido_clean["Area_Metropolitana"].eq("otros").sum()))


                            archivo columnas_municipales_candidatas
0  caracteristicasgenerales_oct.CSV                         ninguna
1  caracteristicasgenerales_nov.CSV                         ninguna
2  caracteristicasgenerales_dic.CSV                         ninguna
3                  ocupados_oct.CSV                         ninguna
4                  ocupados_nov.CSV                         ninguna
5                  ocupados_dic.CSV                         ninguna
Columna municipal en el final: []
Registros de Cali A.M. en el final: 3349
AREA sin identificación específica, etiquetada como otros: 24143


**Resultado observado:** los seis CSV de origen y el final carecen de una variable municipal utilizable. El prototipo de Cali A.M. contiene 3.349 ocupados; 24.143 filas del final tienen AREA igual a "otros". **Validación municipal: pendiente por falta de acceso a un identificador de municipio.** El alcance de Cali no está satisfecho con este producto público.


## 3. Interpretación temporal del cuarto trimestre de 2025

Octubre, noviembre y diciembre de 2025 son tres cortes mensuales. Se agrupan para describir el **promedio del trimestre octubre-diciembre de 2025**. No son tres visitas longitudinales a las mismas personas y la diferencia entre meses no demuestra cambios individuales ni causalidad.

Para una proporción trimestral se divide la suma ponderada del numerador entre la suma ponderada del denominador de los tres meses. Para una **cantidad expandida promedio mensual del trimestre**, se divide entre tres la suma de los factores mensuales; sumarlos sin ese ajuste contaría tres poblaciones mensuales. En una razón de sumas, dividir ambos lados entre tres no altera el cociente. La mediana ponderada se calcula sobre las observaciones de los tres meses y se interpreta como distribución agrupada del trimestre, no como promedio de tres medianas mensuales.


In [4]:
meses = (
    df_traducido_clean.groupby(["Anio_Encuesta", "Mes_Encuesta"])
    .agg(n_ocupados=("Id_Vivienda", "size"),
         suma_factor=("Factor_Expansion", "sum"))
    .reset_index()
    .sort_values(["Anio_Encuesta", "Mes_Encuesta"])
)
display(meses)
periodos = set(zip(meses["Anio_Encuesta"], meses["Mes_Encuesta"]))
print("Octubre-diciembre 2025 completos:",
      periodos == {(2025, 10), (2025, 11), (2025, 12)})
print("Suma de factores / 3 (promedio mensual del conjunto de ocupados):",
      round(meses["suma_factor"].sum() / 3))


   Anio_Encuesta  Mes_Encuesta  n_ocupados   suma_factor
0           2025            10       30807  2.436961e+07
1           2025            11       30939  2.459864e+07
2           2025            12       29611  2.422365e+07
Octubre-diciembre 2025 completos: True
Suma de factores / 3 (promedio mensual del conjunto de ocupados): 24397303


**Resultado observado:** los 91.357 ocupados se distribuyen en 30.807 de octubre, 30.939 de noviembre y 29.611 de diciembre. La suma de factores de cada mes ronda 24,2 a 24,6 millones para el conjunto nacional de ocupados. Su suma trimestral no se interpreta como población simultánea.


## 4. Calidad de las variables necesarias para los KPI

La clasificación de manufactura utiliza el **código original** RAMA2D_R4 de Ocupados: divisiones 10 a 33. Se recupera del CSV transformado mediante la misma clave, porque la columna final ya convirtió los códigos no listados en el diccionario a "otras ramas". Esa etiqueta no debe usarse como prueba de un sector específico. Los demás códigos de dos dígitos de 01 a 99 se agrupan como "otros sectores"; los códigos fuera de ese dominio quedan sin clasificar.

Definición operativa para el prototipo: **asalariados** = códigos P6430 1, 2, 3 y 7 (empleado particular, empleado público, empleado doméstico y jornalero); **cuenta propia** = código 4. Patrón/empleador y otras posiciones quedan fuera de los cuatro grupos comparables. Se exige edad numérica de 18 años o más y factor de expansión positivo. Esta definición deberá documentarse y aprobarse en el diccionario de datos final.


In [5]:
codigos = ocupados[CLAVES + [
    "Rama_Act_Empleo_Ppal", "Categoria_Trabajo", "Area_Metropolitana"
]].rename(columns={
    "Rama_Act_Empleo_Ppal": "rama_codigo",
    "Categoria_Trabajo": "posicion_codigo",
    "Area_Metropolitana": "area_codigo",
})
base = df_traducido_clean.merge(
    codigos, on=CLAVES, how="left", validate="one_to_one", indicator=True
)
base["edad_num"] = pd.to_numeric(base["Edad"], errors="coerce")
base["factor_num"] = pd.to_numeric(base["Factor_Expansion"], errors="coerce")
base["rama_codigo"] = pd.to_numeric(base["rama_codigo"], errors="coerce")
base["posicion_codigo"] = pd.to_numeric(base["posicion_codigo"], errors="coerce")
base["area_codigo"] = pd.to_numeric(base["area_codigo"], errors="coerce")
base["sector"] = np.select(
    [base["rama_codigo"].between(10, 33),
     base["rama_codigo"].between(1, 99)],
    ["Manufactura", "Otros sectores"], default="Sin código válido"
)
base["posicion"] = np.select(
    [base["posicion_codigo"].isin([1, 2, 3, 7]),
     base["posicion_codigo"].eq(4)],
    ["Asalariados", "Cuenta propia"], default="Fuera del alcance"
)
calidad = pd.Series({
    "sin_pareja_en_codigos": int(base["_merge"].ne("both").sum()),
    "clave_nula": int(base[CLAVES].isna().any(axis=1).sum()),
    "edad_no_numerica": int(base["edad_num"].isna().sum()),
    "edad_menor_18": int(base["edad_num"].lt(18).sum()),
    "factor_nulo_no_numerico_o_no_positivo": int(
        (base["factor_num"].isna() | base["factor_num"].le(0)).sum()
    ),
    "rama_sin_codigo_valido": int(base["sector"].eq("Sin código válido").sum()),
    "posicion_fuera_alcance": int(base["posicion"].eq("Fuera del alcance").sum()),
    "ingreso_laboral_nulo": int(
        pd.to_numeric(base["Ingreso_Laboral"], errors="coerce").isna().sum()
    ),
    "rama_etiquetada_otras_ramas": int(
        base["Rama_Act_Empleo_Ppal"].eq("otras ramas").sum()
    ),
})
display(calidad.rename("filas").to_frame())


                                       filas
sin_pareja_en_codigos                      0
clave_nula                                 0
edad_no_numerica                           0
edad_menor_18                            696
factor_nulo_no_numerico_o_no_positivo      0
rama_sin_codigo_valido                     1
posicion_fuera_alcance                  3714
ingreso_laboral_nulo                    1888
rama_etiquetada_otras_ramas            82799


**Resultado observado:** no hay claves nulas, factores inválidos ni filas del final sin pareja en Ocupados. Hay 1.888 ingresos laborales nulos. La columna final contiene 82.799 etiquetas "otras ramas" porque el catálogo de traducción solo enumera manufactura; para separar manufactura del resto se usa el código original. El ingreso no interviene en los tres KPI de relevo.


## 5. Indicadores y advertencias por tamaño de muestra

Las tablas siguientes son **solo un prototipo de Cali A.M. (incluye Yumbo)**. La proporción de 55+ usa como denominador a las personas de 18+ del grupo. La razón de reemplazo compara sumas ponderadas de 18–34 y 55+; se deja indefinida si no hay peso en 55+. La mediana ponderada es la primera edad cuya suma acumulada de factores alcanza al menos la mitad del peso del grupo.

Regla de advertencia interna, no umbral oficial de precisión: menos de 30 observaciones en un grupo o en un componente de la razón = **crítico**; entre 30 y 99 = **cautela**; 100 o más = **sin alerta por n**. El tamaño no ponderado y el tamaño efectivo aproximado de Kish son diagnósticos, **no** sustituyen error estándar, intervalo de confianza ni coeficiente de variación del diseño complejo. Sin estratos y conglomerados no se establece precisión oficial.


In [6]:
def mediana_ponderada(edades, factores):
    orden = np.argsort(np.asarray(edades))
    e = np.asarray(edades)[orden]
    w = np.asarray(factores)[orden]
    return float(e[np.searchsorted(np.cumsum(w), w.sum() / 2, side="left")])

def alerta_n(n):
    if n < 30:
        return "CRÍTICO: n < 30"
    if n < 100:
        return "CAUTELA: 30 <= n < 100"
    return "Sin alerta por n"

piloto = base[
    base["area_codigo"].eq(76)
    & base["edad_num"].ge(18)
    & base["factor_num"].gt(0)
    & base["sector"].ne("Sin código válido")
    & base["posicion"].ne("Fuera del alcance")
].copy()
filas = []
for (sector, posicion), grupo in piloto.groupby(["sector", "posicion"]):
    joven = grupo["edad_num"].between(18, 34)
    mayor = grupo["edad_num"].ge(55)
    peso_total = grupo["factor_num"].sum()
    peso_joven = grupo.loc[joven, "factor_num"].sum()
    peso_mayor = grupo.loc[mayor, "factor_num"].sum()
    n_efectivo_aprox = peso_total**2 / grupo["factor_num"].pow(2).sum()
    filas.append({
        "área": "Cali A.M. (incluye Yumbo)",
        "sector": sector,
        "posición": posicion,
        "n_grupo": len(grupo),
        "n_18_34": int(joven.sum()),
        "n_55_más": int(mayor.sum()),
        "n_efectivo_aprox": round(n_efectivo_aprox, 1),
        "proporción_55_más_pct": round(100 * peso_mayor / peso_total, 1),
        "razón_reemplazo": round(peso_joven / peso_mayor, 2) if peso_mayor > 0 else np.nan,
        "edad_mediana_ponderada": mediana_ponderada(
            grupo["edad_num"], grupo["factor_num"]
        ),
        "alerta_grupo": alerta_n(len(grupo)),
        "alerta_razón_18_34": alerta_n(int(joven.sum())),
        "alerta_razón_55_más": alerta_n(int(mayor.sum())),
    })
indicadores_piloto = pd.DataFrame(filas).sort_values(["sector", "posición"])
display(indicadores_piloto)


                        área          sector       posición  n_grupo  n_18_34  n_55_más  n_efectivo_aprox  proporción_55_más_pct  razón_reemplazo  edad_mediana_ponderada            alerta_grupo alerta_razón_18_34     alerta_razón_55_más
0  Cali A.M. (incluye Yumbo)     Manufactura    Asalariados      428      175        48             307.9                   10.0             4.07                    38.0        Sin alerta por n   Sin alerta por n  CAUTELA: 30 <= n < 100
1  Cali A.M. (incluye Yumbo)     Manufactura  Cuenta propia       90       13        34              72.9                   33.3             0.41                    46.0  CAUTELA: 30 <= n < 100    CRÍTICO: n < 30  CAUTELA: 30 <= n < 100
2  Cali A.M. (incluye Yumbo)  Otros sectores    Asalariados     1684      701       261            1349.0                   14.5             2.90                    37.0        Sin alerta por n   Sin alerta por n        Sin alerta por n
3  Cali A.M. (incluye Yumbo)  Otros sectores  Cuenta

**Lectura del prototipo inspeccionado:** manufactura asalariada tiene 428 observaciones (48 de 55+); manufactura por cuenta propia, 90 observaciones de 18+ (13 de 18–34 y 34 de 55+). La razón de reemplazo de manufactura por cuenta propia recibe alerta **crítica** por sus 13 observaciones jóvenes. Las cifras ponderadas del piloto pueden ayudar a depurar el método, pero no responden todavía la pregunta municipal del proyecto. Para publicar un resultado de Cali hace falta el acceso municipal y una evaluación de precisión compatible con el diseño muestral.


## Decisión de avance y fuentes

1. **Unión persona-mes:** aprobada con los archivos actuales; ninguna pérdida de ocupados y ninguna duplicación.
2. **Cobertura de Cali municipio:** bloqueada en la fuente pública. Solicitar o incorporar el identificador municipal del acceso licenciado/SPEE y repetir los controles.
3. **Tiempo:** tratar octubre–diciembre de 2025 como tres cortes mensuales agrupados para el cuarto trimestre. Las cantidades expandidas del trimestre se expresan como promedio mensual.
4. **Calidad:** mantener la rama original de dos dígitos y la posición ocupacional para clasificar grupos; excluir registros sin edad válida, factor positivo o código interpretable; publicar n y advertencias con cada KPI.

**Fuentes:** documento de alcance del proyecto, Avance 1 Proyecto Diligenciado Minimo2 final (1).docx, secciones Contexto, Fuentes de datos y OKR/KPI; [DANE, catálogo GEIH 2025](https://microdatos.dane.gov.co/index.php/catalog/853); [DANE, variable AREA](https://microdatos.dane.gov.co/index.php/catalog/853/variable/F1/V9?name=AREA); [DANE, gu?a de acceso y uso de bases anonimizadas GEIH](https://microdatos.dane.gov.co/catalog/750/download/21521/Acceso_y_Uso_de_Bases_Anonimizadas-GEIH.pdf). La regla de alerta por n y la clasificación de asalariados son definiciones operativas de este análisis, no criterios oficiales del DANE.
